# 具備 Human-in-the-Loop 的 Agents

我們已經有一個 email assistant，它用 router 來對 email 進行 triage，然後把 email 交給 agent 產生回應。我們也已經評估過它了。但我們真的*信任*它能自主管理我們的收件匣嗎？對於這麼敏感的任務，human-in-the-loop（HITL）非常重要！在這裡，我們會示範如何為 email assistant 加上 human-in-the-loop，讓我們能審查特定的 tool calls。

![overview-img](img/overview_hitl.png)


我們將示範如何讓 graph 在特定的點上*暫停*下來，並等待人類的輸入。

![overview-img](img/hitl_schematic.png)

#### 載入環境變數

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")

## 為我們的 email assistant 加上 HITL

我們來為 email assistant 加上 HITL。

我們可以像之前一樣，從 tools 開始。

但現在，我們會新增一個 Question tool，讓 assistant 能向使用者提出問題。

In [ ]:

%load_ext autoreload
%autoreload 2

from typing import Literal
from datetime import datetime
from pydantic import BaseModel

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
# interrupt 是本章的主角。⚠️ 它和你可能在別的教材看過的 interrupt_before 是「兩套」東西：
#   interrupt_before=["node"] → compile 時就定死，停在「進 node 之前」。
#                               人類只能從外面改 state，然後用 stream(None, ...) 續跑。
#   interrupt([request])      → 寫在 node「裡面」，停在這一行。
#                               人類的答案會「變成這一行的回傳值」，用 Command(resume=[...]) 送進來。
# 本系列全部採用後者：暫停點寫在程式碼流裡，讀起來像一個會停很久的 input()。
# ⚠️ 兩套 API 混用會很亂（設定的地方不同、續跑的傳法也不同），選一套就別回頭。
from langgraph.types import interrupt, Command

from email_assistant.prompts import triage_system_prompt, triage_user_prompt, agent_system_prompt_hitl, default_background, default_triage_instructions, default_response_preferences, default_cal_preferences
from email_assistant.tools.default.prompt_templates import HITL_TOOLS_PROMPT
from email_assistant.schemas import State, RouterSchema, StateInput
from email_assistant.utils import parse_email, format_for_display, format_email_markdown

# Agent tools
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """撰寫並寄出一封 email。"""
    # 佔位回應 - 真實應用中會實際寄出 email
    return f"Email sent to {to} with subject '{subject}' and content: {content}"

@tool
def schedule_meeting(
    attendees: list[str], subject: str, duration_minutes: int, preferred_day: datetime, start_time: int
) -> str:
    """在行事曆上排定一場會議。"""
    # 佔位回應 - 真實應用中會檢查行事曆並排程
    date_str = preferred_day.strftime("%A, %B %d, %Y")
    return f"Meeting '{subject}' scheduled on {date_str} at {start_time} for {duration_minutes} minutes with {len(attendees)} attendees"

@tool
def check_calendar_availability(day: str) -> str:
    """查詢某一天的行事曆空檔。"""
    # 佔位回應 - 真實應用中會檢查實際的行事曆
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

@tool
# 這是新增的！而且它是本章的關鍵設計。
# ⚠️ Question 沒有任何實作，它「不是拿來執行的」——它是拿來被攔截的。
# LLM 想問人問題時，就開一張 Question 工單，interrupt_handler 攔下來轉成 interrupt。
# 意義：把「跟人講話」降格成跟「寄信」同一種東西——一張 tool call 工單，
# 於是它自動共用同一套審核機制，不必為了「提問」另外寫一條流程。
class Question(BaseModel):
      """要詢問使用者的問題。"""
      content: str
    
@tool
class Done(BaseModel):
      """Email 已寄出。"""
      done: bool

# agent 可用的所有 tools
tools = [
    write_email, 
    schedule_meeting, 
    check_calendar_availability, 
    Question, 
    Done,
]

tools_by_name = {tool.name: tool for tool in tools}

# 初始化要搭配 router / structured output 使用的 LLM。
# （下面第 2 次 init_chat_model 的參數和這裡一模一樣，是教材的冗餘，不是有什麼玄機。）
llm = init_chat_model("openai:gpt-4.1", temperature=0.0)
llm_router = llm.with_structured_output(RouterSchema) 

# tool_choice="required" = 強迫 LLM 每一輪都必須開至少一張工單，不准只回一段純文字。
# 為什麼要強迫？因為這個 agent 的「我講完了」是用 Done 這張工單來表示的（見 should_continue）。
# ⚠️ 這行不只是風格選擇，它在幫下面的 should_continue 擦屁股：
#    should_continue 沒有處理「LLM 回純文字」的情況，真的發生會直接拋錯。
#    拿掉 required，那個洞就會露出來。
llm = init_chat_model("openai:gpt-4.1", temperature=0.0)
llm_with_tools = llm.bind_tools(tools, tool_choice="required")

In [ ]:
from rich.markdown import Markdown
Markdown(HITL_TOOLS_PROMPT)

#### Triage node

我們定義一個帶有 triage 路由邏輯的 Python 函式，就像之前一樣。

但是，如果分類結果是 `notify`，我們會想中斷 graph，好讓使用者來審查這封 email！

所以我們會前往一個新的 node，`triage_interrupt_handler`。

In [26]:
def triage_router(state: State) -> Command[Literal["triage_interrupt_handler", "response_agent", "__end__"]]:
    """分析 email 內容，判斷我們應該回覆、通知、還是忽略。"""

    # 解析 email 輸入
    author, to, subject, email_thread = parse_email(state["email_input"])
    user_prompt = triage_user_prompt.format(
        author=author, to=to, subject=subject, email_thread=email_thread
    )

    # 為 Agent Inbox 製作 email markdown，以備需要通知時使用
    email_markdown = format_email_markdown(subject, author, to, email_thread)

    # 用 background 與 triage 指示來格式化 system prompt
    system_prompt = triage_system_prompt.format(
        background=default_background,
        triage_instructions=default_triage_instructions
    )

    # 執行 router LLM
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )

    # 判斷結果
    classification = result.classification

    # 處理分類判斷的結果
    if classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        # 下一個 node
        goto = "response_agent"
        # 更新 state
        update = {
            "classification_decision": classification,
            "messages": [{"role": "user",
                            "content": f"Respond to the email: {email_markdown}"
                        }],
        }
    elif classification == "ignore":
        print("🚫 Classification: IGNORE - This email can be safely ignored")
        # 下一個 node
        goto = END
        # 更新 state
        update = {
            "classification_decision": classification,
        }

    elif classification == "notify":
        print("🔔 Classification: NOTIFY - This email contains important information") 
        # 這是第 4 章對 triage_router 唯一的改動。
        # 前面幾章的 notify 是直接 END（「知道了，但沒事做」），現在改成「停下來拿給人看」。
        # 注意這裡的 update 沒有把 email 塞進 messages——那是 triage_interrupt_handler 的事，
        # 因為使用者可能選擇忽略，這封信就根本不該進入對話歷史。
        goto = "triage_interrupt_handler"
        # 更新 state
        update = {
            "classification_decision": classification,
        }

    else:
        raise ValueError(f"Invalid classification: {classification}")
    # 同第 2 章：node 自己決定下一步去哪，不靠外面的 conditional edge。
    return Command(goto=goto, update=update)

#### Triage Interrupt Handler

如果決策是要 `notify`（通知）使用者，我們就中斷 graph！

![overview-img](img/HITL_flow_triage.png)

為此，我們加入一個新的 node，`triage_interrupt_handler`，它將會：

1. 若分類為 `notify`，把分類結果顯示給使用者：我們會把一個帶有分類結果的 `dict` 傳給 interrupt。
2. 讓使用者對這個決策做出回應：我們會設計程式碼來處理從 Agent Inbox 收回來的內容。

正如你在[這裡](https://github.com/langchain-ai/agent-inbox?tab=readme-ov-file#what-do-the-fields-mean)所見，我們用特定的欄位來格式化 interrupt，好讓它能在 Agent Inbox 中被檢視：

* `action_request`：此次 interrupt 的動作與參數，包含 `action`（動作名稱）與 `args`（tool call 的參數）。它會在 Agent Inbox 中被算繪為這次 interrupt 事件的主標題。
* `config`：設定允許哪些互動類型，以及各類型對應的特定 UI 元素。
* `description`：應該寫得詳細，並可使用 markdown。它會在 Agent Inbox 中被算繪為描述。

In [27]:
def triage_interrupt_handler(state: State) -> Command[Literal["response_agent", "__end__"]]:
    """處理 triage 步驟產生的 interrupts。"""
    
    # ⚠️ 踩雷（本章最重要的一條，整章都靠它，實測驗證過）：
    # 被 Command(resume=...) 恢復時，這個 node 是「從第一行重新執行」，
    # 不是從 interrupt() 那一行接著跑。
    #
    # 為什麼？因為 LangGraph 沒有把 Python 的執行堆疊冷凍起來（那做不到）。
    # 它是把整個 node 當成一個「可重放的函式」：重跑一次，跑到 interrupt() 時，
    # 直接把你 resume 的值當成回傳值塞回去（已經回答過的 interrupt 不會再問第二次）。
    #
    # 後果：interrupt() 「前面」的每一行，都會跑第二次。
    # 這裡只是 parse email、組字串，重跑無害——這正是為什麼教材把 interrupt()
    # 放在 node 很前面的位置，而且前面只放純計算。
    # 如果你在 interrupt() 前面寄信、寫 DB、扣款，它就會做兩次。
    # 解析 email 輸入
    author, to, subject, email_thread = parse_email(state["email_input"])

    # 為 Agent Inbox 製作 email markdown，以備需要通知時使用
    email_markdown = format_email_markdown(subject, author, to, email_thread)

    # 建立 messages
    messages = [{"role": "user",
                "content": f"Email to notify user about: {email_markdown}"
                }]

    # 建立要顯示給使用者的 interrupt
    # ⚠️ 這個 dict 不是隨便長的，它是 Agent Inbox UI 的「約定格式」，key 名一個都不能改。
    # 你送什麼進去，決定使用者在畫面上看到什麼、有哪些按鈕可以按：
    #   action_request.action → 卡片的大標題
    #   action_request.args   → 標題底下列出的參數（這裡是純通知，沒有參數要審，所以空的）
    #   description           → 卡片內文，吃 markdown
    #   config.allow_*        → 哪幾顆按鈕會亮
    request = {
        "action_request": {
            "action": f"Email Assistant: {state['classification_decision']}",
            "args": {}
        },
        # 這是「通知」不是「動作」：沒有 tool call 要核准，所以 accept／edit 兩顆按鈕關掉。
        # 使用者只有兩條路可走：忽略掉，或講一句話叫 agent 去回信。
        "config": {
            "allow_ignore": True,  
            "allow_respond": True, 
            "allow_edit": False, 
            "allow_accept": False,  
        },
        # 要在 Agent Inbox 中顯示的 email
        "description": email_markdown,
    }

    # 執行到這一行，整張 graph 就地凍結，request 被丟出去給外面的人看。
    #
    # 為什麼傳「list」進去、又立刻取「[0]」出來？
    # 因為 interrupt 的協定設計成「一次可以送出多個請求、收回多個回應」，所以
    # 進去是 list、回來也是 list。這裡只送一個請求，就取第 0 個回應。
    # 恢復端要對應：Command(resume=[{"type": "response", "args": "..."}]) 也是 list。
    #
    # 回傳值是 dict，只有一個 key `type`，值為 accept / edit / ignore / response 四選一
    # （這裡的 config 只開了後兩種）。人類的答案「變成這一行的回傳值」——
    # 這就是 interrupt 和 interrupt_before 最本質的差別。
    response = interrupt([request])[0]

    # response = 使用者不想忽略，他打了一句話叫 agent 去回這封信。
    # 那句話會變成一則 user message，接力給 response agent 當作指示。
    if response["type"] == "response":
        # 把回饋加入 messages
        user_input = response["args"]
        # 供 response agent 使用
        messages.append({"role": "user",
                        "content": f"User wants to reply to the email. Use this feedback to respond: {user_input}"
                        })
        # 前往 response agent
        goto = "response_agent"

    # ignore = 看過了，不用處理。整條 workflow 到此為止，這封信不會進入 response agent。
    elif response["type"] == "ignore":
        goto = END

    # 攔截所有其他回應
    else:
        raise ValueError(f"Invalid response: {response}")

    # 更新 state
    update = {
        "messages": messages,
    }

    return Command(goto=goto, update=update)

#### LLM call

`llm_call` 這個 node 與之前相同：

In [28]:
def llm_call(state: State):
    """由 LLM 決定要不要呼叫 tool。"""

    return {
        "messages": [
            llm_with_tools.invoke(
                [
                    {"role": "system", "content": agent_system_prompt_hitl.format(tools_prompt=HITL_TOOLS_PROMPT, 
                                                                                  background=default_background,
                                                                                  response_preferences=default_response_preferences, 
                                                                                  cal_preferences=default_cal_preferences)}
                ]
                + state["messages"]
            )
        ]
    }

#### Interrupt Handler

`interrupt_handler` 是我們 response agent 中核心的 HITL 元件。

它的工作是檢查 LLM 想要進行的 tool calls，並判斷其中哪些需要在執行前經過人工審查。它的運作方式如下：

1. **Tool Selection（選擇 tool）**：這個 handler 維護一份「HITL tools」清單，列出需要人工核准的 tools：
   - `write_email`：因為寄出 email 會造成顯著的外部影響
   - `schedule_meeting`：因為安排會議會影響行事曆
   - `Question`：因為向使用者提問需要直接的互動

2. **Direct Execution（直接執行）**：不在 HITL 清單中的 tools（例如 `check_calendar_availability`）會立即執行而不中斷。這讓低風險的操作能自動進行。

3. **Context Preparation（準備情境）**：對於需要審查的 tools，handler 會：
   - 取出原始 email 以提供情境
   - 把 tool call 的細節格式化以便清楚顯示
   - 設定各類 tool 允許哪些互動類型

4. **Interrupt Creation（建立 interrupt）**：handler 會建立一個結構化的 interrupt request，其中包含：
   - 動作名稱與參數
   - 允許哪些互動類型的設定
   - 一段同時包含原始 email 與所提議動作的描述

5. **Response Processing（處理回應）**：在 interrupt 之後，handler 會處理人類的回應：
   - **Accept**：用原始參數執行該 tool
   - **Edit**：用編輯後的參數更新 tool call，然後執行
   - **Ignore**：取消該 tool 的執行
   - **Response**：記錄回饋但不執行

這個 handler 確保人類能監督所有重要的動作，同時讓例行操作能自動進行。

能夠編輯 tool 參數（例如 email 內容或會議細節）讓使用者得以精準掌控 assistant 的動作。

我們可以把整體流程視覺化呈現：

![overview-img](img/HITL_flow.png)

In [29]:
def interrupt_handler(state: State) -> Command[Literal["llm_call", "__end__"]]:
    """建立 interrupt，讓人工審查 tool calls"""
    
    # ⚠️ 踩雷（承 triage_interrupt_handler，這裡後果嚴重得多，已實測）：
    # 這個 node 每被 resume 一次就從頭重跑一次，這個 for 迴圈也會從第 0 張工單重數。
    # 已經審過的工單，其 interrupt() 會把上次的答案「重放」回來（不會再問你一次），
    # 但是——它前面那行 tool.invoke() 會「真的再執行一次」。
    #
    # 實測（3 張工單 check_calendar + schedule_meeting + write_email，全部 accept）：
    #   check_calendar   被執行 3 次   ← 每次重跑都走一遍直接執行的路
    #   schedule_meeting 被執行 2 次   ← 第 2 輪被 accept 執行，第 3 輪重跑又執行一次
    #   write_email      被執行 1 次   ← 最後一輪才輪到它
    #
    # ⚠️ 最陰的地方：最終 state 完全正常（每張工單剛好一則 tool message），
    #    因為中斷那幾輪的 return 根本沒被 commit（下面 result 每輪都是全新的空 list）。
    #    也就是說「重複執行」在 state 裡完全看不出來，只有真的副作用會露餡。
    #    本教材沒出事，純粹因為這些 tool 都是回傳字串的假貨。
    #    上線前的解法：讓 tool 具備冪等性（idempotent），或一次只審一張工單。
    #
    # 暫存 messages（每輪重跑都會被清空重來，所以不會累積重複）
    result = []

    # 預設回 llm_call，讓 LLM 看到 tool 結果後決定下一步（再開工單，或開 Done 收工）。
    # 下面只有 ignore 分支會把 goto 改成 END——使用者說不要，就沒必要再問 LLM 了。
    goto = "llm_call"

    # 逐一處理最後一則 message 中的 tool calls
    for tool_call in state["messages"][-1].tool_calls:
        
        # 哪些 tool 要人審、哪些不用，全靠這一行。判準是「做了會不會對外造成後果」：
        #   write_email / schedule_meeting → 信真的寄出去、行事曆真的被佔位 → 要審
        #   Question                       → 本來就是要問人的 → 要審
        #   check_calendar_availability    → 只是讀資料，做錯了也不痛 → 讓它自己跑
        hitl_tools = ["write_email", "schedule_meeting", "Question"]
        
        # 若 tool 不在我們的 HITL 清單中，直接執行、不中斷
        if tool_call["name"] not in hitl_tools:

            # ⚠️ 這裡是上面那條踩雷的第一現場：node 每重跑一次，這個 tool 就被 invoke 一次。
            # 只有唯讀的 tool 放得進這條路，會改東西的絕不能放這裡。
            tool = tools_by_name[tool_call["name"]]
            observation = tool.invoke(tool_call["args"])
            result.append({"role": "tool", "content": observation, "tool_call_id": tool_call["id"]})
            continue
            
        # 從 state 的 email_input 取得原始 email
        email_input = state["email_input"]
        author, to, subject, email_thread = parse_email(email_input)
        original_email_markdown = format_email_markdown(subject, author, to, email_thread)
        
        # 格式化 tool call 以供顯示，並在前面附上原始 email
        tool_display = format_for_display(tool_call)
        description = original_email_markdown + tool_display

        # 每種 tool 給人類的權限不一樣，這裡就是在決定 Agent Inbox 上哪幾顆按鈕會亮。
        # write_email / schedule_meeting：四顆全開（可接受、可改參數、可回饋、可忽略）。
        # Question：只給 ignore／respond——「問題」沒有參數可以編輯，「接受一個問題」也不知道是什麼意思。
        if tool_call["name"] == "write_email":
            config = {
                "allow_ignore": True,
                "allow_respond": True,
                "allow_edit": True,
                "allow_accept": True,
            }
        elif tool_call["name"] == "schedule_meeting":
            config = {
                "allow_ignore": True,
                "allow_respond": True,
                "allow_edit": True,
                "allow_accept": True,
            }
        elif tool_call["name"] == "Question":
            config = {
                "allow_ignore": True,
                "allow_respond": True,
                "allow_edit": False,
                "allow_accept": False,
            }
        else:
            raise ValueError(f"Invalid tool call: {tool_call['name']}")

        # 建立 interrupt request
        request = {
            "action_request": {
                "action": tool_call["name"],
                "args": tool_call["args"]
            },
            "config": config,
            "description": description,
        }

        # 同 triage_interrupt_handler：送 list 進去、取 [0] 出來，人類的答案就是這行的回傳值。
        # 差別是這次 request 帶了真正的 tool call 參數（action_request.args），
        # 所以 Agent Inbox 上會長出可以直接改的欄位——這就是 edit 之所以可能的前提。
        response = interrupt([request])[0]

        # 處理各種回應
        if response["type"] == "accept":

            # accept = 使用者按了「同意」。一個字都不改，用 LLM 原本開的參數執行。
            # 這是四條路裡唯一「什麼都沒發生過」的一條：對話歷史和沒有人類介入時一模一樣。
            tool = tools_by_name[tool_call["name"]]
            observation = tool.invoke(tool_call["args"])
            result.append({"role": "tool", "content": observation, "tool_call_id": tool_call["id"]})
                        
        elif response["type"] == "edit":

            # edit = 使用者把參數改過了才放行。以下這段是本章技術含量最高的地方：
            # 我們要「竄改歷史」——把 LLM 原本開的那張工單，換成人類改過的版本，
            # 讓後面的流程以為 LLM 本來就是這樣說的。
            #
            # 為什麼非改不可？如果留著 LLM 的原版工單、卻執行人類的參數，對話紀錄就和現實對不起來：
            # 下一輪 llm_call 會讀到「我提議開 45 分鐘」＋「會議已排定」，
            # 於是它會拿著 45 分鐘這個錯誤前提去寫信，而實際上排的是 30 分鐘。
            tool = tools_by_name[tool_call["name"]]
            
            # 從 Agent Inbox 取得編輯後的 args
            edited_args = response["args"]["args"]

            # 用編輯後的內容更新 AI message 的 tool call（指向 state 中的該則 message）
            ai_message = state["messages"][-1] # 從 state 取得最新的一則 message
            current_id = tool_call["id"] # 儲存正在編輯的那個 tool call 的 ID
            
            # 濾掉舊的那一張、補上改過的那一張。注意這是「重建一個新 list」，
            # 而不是在原 list 上改值——原 list 屬於已經存進 checkpoint 的那顆 message，不該碰。
            # 副作用：被編輯的那張工單會被排到 list 最後面（順序變了）。這裡無所謂，因為配對靠 id 不靠位置。
            updated_tool_calls = [tc for tc in ai_message.tool_calls if tc["id"] != current_id] + [
                {"type": "tool_call", "name": tool_call["name"], "args": edited_args, "id": current_id}
            ]
            
            # 這一行就是「竄改歷史」的實際動作。兩個關鍵：
            #
            # 1. 為什麼用 model_copy，不直接 ai_message.tool_calls = updated_tool_calls？
            #    因為這顆 message 已經躺在 checkpoint 裡了，要當成不可變的東西看待。
            #    直接改它等於偷改「已經發生的過去」，之後 time travel 回去看會看到被污染的歷史。
            #    model_copy 產生的是一顆全新的 message，但 id 保持相同——這點是下一段的關鍵。
            #
            # 2. 為什麼「append 一顆新 message」可以達成「修改舊 message」？
            #    因為 messages 這個 state key 掛的是 add_messages reducer，它是「依 id 比對」的：
            #      id 相同 → 覆蓋掉舊的那顆（list 長度不變）
            #      id 不同 → 當成新的一顆，接在後面（list 變長）
            #    我們要的是覆蓋，所以上面才要死守 current_id。
            #    ⚠️ 踩雷：id 一換，就變成憑空多一則 AI message，而原本那張錯的工單還留在歷史裡，
            #       LLM 下一輪會看到兩張工單、開始鬼打牆——而且不會有任何錯誤訊息。
            result.append(ai_message.model_copy(update={"tool_calls": updated_tool_calls}))

            # 用 Agent Inbox 編輯後的內容更新 write_email 的 tool call
            if tool_call["name"] == "write_email":
                
                # 用編輯後的 args 執行 tool
                observation = tool.invoke(edited_args)
                
                # 只加入 tool 回應的 message
                result.append({"role": "tool", "content": observation, "tool_call_id": current_id})
            
            # 用 Agent Inbox 編輯後的內容更新 schedule_meeting 的 tool call
            elif tool_call["name"] == "schedule_meeting":
                
                
                # 用編輯後的 args 執行 tool
                observation = tool.invoke(edited_args)
                
                # 只加入 tool 回應的 message
                result.append({"role": "tool", "content": observation, "tool_call_id": current_id})
            
            # 攔截所有其他 tool calls
            else:
                raise ValueError(f"Invalid tool call: {tool_call['name']}")

        # ignore = 使用者按了「忽略」。tool 不執行，而且整條 workflow 直接收工（goto = END）。
        # ⚠️ 注意這裡「還是」塞了一則 tool message 回去，而不是什麼都不做——
        # 因為 tool calling 的協定規定：每一張 tool call 都必須有一則對應的 tool message，
        # 少一則，下次把 messages 送進 LLM 就會被 API 打回票。
        # （內容寫成一句給 LLM 看的指令，但這裡 goto=END，LLM 其實已經不會再讀到它了。）
        elif response["type"] == "ignore":
            if tool_call["name"] == "write_email":
                # 不執行 tool，並告訴 agent 接下來該怎麼做
                result.append({"role": "tool", "content": "User ignored this email draft. Ignore this email and end the workflow.", "tool_call_id": tool_call["id"]})
                # 前往 END
                goto = END
            elif tool_call["name"] == "schedule_meeting":
                # 不執行 tool，並告訴 agent 接下來該怎麼做
                result.append({"role": "tool", "content": "User ignored this calendar meeting draft. Ignore this email and end the workflow.", "tool_call_id": tool_call["id"]})
                # 前往 END
                goto = END
            elif tool_call["name"] == "Question":
                # 不執行 tool，並告訴 agent 接下來該怎麼做
                result.append({"role": "tool", "content": "User ignored this question. Ignore this email and end the workflow.", "tool_call_id": tool_call["id"]})
                # 前往 END
                goto = END
            else:
                raise ValueError(f"Invalid tool call: {tool_call['name']}")
            
        # response = 使用者不接受也不改，而是「講一句話」。tool 不執行，
        # 把那句話包成 tool message 丟回去，goto 維持 llm_call，讓 LLM 自己讀懂回饋、重開一張新工單。
        #
        # 對照組（這章最該講清楚的一組）：
        #   edit     = 「你別想了，照我寫的做」→ 當場執行，LLM 沒有再思考的機會
        #   response = 「你再想一次，往這個方向」→ 回到 LLM 手上，它會重開一張工單，於是要再審一次
        elif response["type"] == "response":
            # 使用者提供了回饋
            user_feedback = response["args"]
            if tool_call["name"] == "write_email":
                # 不執行 tool，並加入一則帶有使用者回饋的 message，以便納入 email 中
                result.append({"role": "tool", "content": f"User gave feedback, which can we incorporate into the email. Feedback: {user_feedback}", "tool_call_id": tool_call["id"]})
            elif tool_call["name"] == "schedule_meeting":
                # 不執行 tool，並加入一則帶有使用者回饋的 message，以便納入 email 中
                result.append({"role": "tool", "content": f"User gave feedback, which can we incorporate into the meeting request. Feedback: {user_feedback}", "tool_call_id": tool_call["id"]})
            elif tool_call["name"] == "Question": 
                # 不執行 tool，並加入一則帶有使用者回饋的 message，以便納入 email 中
                result.append({"role": "tool", "content": f"User answered the question, which can we can use for any follow up actions. Feedback: {user_feedback}", "tool_call_id": tool_call["id"]})
            else:
                raise ValueError(f"Invalid tool call: {tool_call['name']}")

        # 攔截所有其他回應
        else:
            raise ValueError(f"Invalid response: {response}")
            
    # 更新 state
    update = {
        "messages": result,
    }

    # 同第 2 章：node 自己決定去向。
    # 一路 accept／edit／response → 回 llm_call 繼續；
    # ⚠️ 只要「任何一張」工單被 ignore，goto 就被改成 END 且不會再被改回來——
    #    同一批的其他工單還是會照跑完，但跑完就結束了。
    return Command(goto=goto, update=update)

現在，我們來編譯這張 graph。

In [ ]:
from email_assistant.utils import show_graph

# Conditional edge 函式
# ⚠️ 這個 for 迴圈其實只看得到第一張工單：不管走 if 還是 else 都會 return，第一圈就跳出去了。
#    所以萬一 LLM 一次開了 [write_email, Done] 兩張，Done 會被漏掉。
#    這裡沒出事，是因為 prompt 讓 LLM 一次只開一張。（原始教材如此，不改它，但值得台上點一句。）
# ⚠️ 另外 if 沒有 else：萬一 LLM 回純文字（沒有任何 tool_calls），這函式回傳 None，
#    conditional edge 拿 None 去查表，直接 KeyError。擋住這件事的是前面的
#    tool_choice="required"，不是這裡的程式碼。
def should_continue(state: State) -> Literal["interrupt_handler", "__end__"]:
    """路由到 tool handler，若呼叫了 Done tool 則結束"""
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        for tool_call in last_message.tool_calls: 
            if tool_call["name"] == "Done":
                return END
            else:
                return "interrupt_handler"

# 建立 workflow
agent_builder = StateGraph(State)

# 加入 nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("interrupt_handler", interrupt_handler)

# 加入 edges
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    {
        "interrupt_handler": "interrupt_handler",
        END: END,
    },
)

# ⚠️ 這裡 compile 沒給 checkpointer，但下面 interrupt() 要用。為什麼沒事？
# 因為它等一下是被當成「子 graph」掛進 overall_workflow（.add_node("response_agent", response_agent)），
# 子 graph 會直接沿用母 graph 的 checkpointer。
# 編譯 agent
response_agent = agent_builder.compile()

# 只宣告了 START → triage_router 這一條 edge，其他箭頭一條都沒寫。
# 因為 triage_router / triage_interrupt_handler / interrupt_handler 都是靠
# return Command(goto=...) 自己決定去向的。
# 那 show_graph 怎麼知道要畫哪些箭頭？靠函式簽章上的 Command[Literal["response_agent", "__end__"]]——
# ⚠️ 那個型別註記不只是給人看的型別提示，LangGraph 會真的讀它來畫圖。寫錯了圖就畫錯，但程式照跑。
# 建立整體 workflow
overall_workflow = (
    StateGraph(State, input=StateInput)
    .add_node(triage_router)
    .add_node(triage_interrupt_handler)
    .add_node("response_agent", response_agent)
    .add_edge(START, "triage_router")
    
)

# ⚠️ 這次 compile 一樣沒有 checkpointer，所以這張 graph 只能拿來畫圖，不能真的跑到 interrupt。
# 真正要跑的版本在下一個 cell 會重新 compile 一次，並帶上 InMemorySaver。
email_assistant = overall_workflow.compile()
show_graph(email_assistant, xray=True)

#### HITL 模式回顧

**Triage Interruption（分類中斷）** 當一封 email 被分類為 "notify" 時，系統會中斷以把該 email 顯示給人類使用者
- *User Decision（使用者決策）*：使用者可以選擇忽略這則通知，或提供回饋以回覆該 email
- *Flow Control（流程控制）*：若忽略，workflow 結束；若使用者提供回饋，則流向 Response Agent

**Write Email**：系統會把所提議的 email 草稿顯示給人類審查
- *User Decision and Flow Control*：忽略（結束 workflow）、附帶回饋回應、原樣接受草稿，或編輯草稿

**Schedule Meeting**：系統會把所提議的會議細節顯示給人類審查
- *User Decision and Flow Control*：忽略（結束 workflow）、附帶回饋回應、原樣接受會議細節，或編輯細節

**Question**：系統向使用者提問以釐清資訊
- *User Decision and Flow Control*：忽略（結束 workflow）或以一個答案回應

### Interrupts 讓我們能審查並接受 Tool Calls

In [ ]:
import uuid
from langgraph.checkpoint.memory import InMemorySaver

# 要回覆的 email
email_input_respond = {
    "to": "Lance Martin <lance@company.com>",
    "author": "Project Manager <pm@client.com>",
    "subject": "Tax season let's schedule call",
    "email_thread": "Lance,\n\nIt's tax season again, and I wanted to schedule a call to discuss your tax planning strategies for this year. I have some suggestions that could potentially save you money.\n\nAre you available sometime next week? Tuesday or Thursday afternoon would work best for me, for about 45 minutes.\n\nRegards,\nProject Manager"
}

# ⚠️ 踩雷（會安靜出事的那種）：沒有 checkpointer 時，interrupt() 「不會」報錯。
# 它照樣停、照樣吐給你 __interrupt__，看起來一切正常——
# 直到你要 Command(resume=...) 恢復時才會炸：RuntimeError: Cannot use Command(resume=...) without checkpointer。
# 道理很單純：「暫停」的意思是把現在的 state 存起來、程式退出去，等人回來再撈出來。
# 沒有存檔的地方，就沒有「回來」這回事。上一個 cell 的 compile() 沒帶 checkpointer，所以這裡要重 compile。
checkpointer = InMemorySaver()
graph = overall_workflow.compile(checkpointer=checkpointer)
# 每個示範用一個全新的 thread_id（＝一個全新的存檔），互不干擾。
# ⚠️ 踩雷：若整份 notebook 共用同一個 thread_id，第二個示範會接在第一個示範的 messages
# 後面繼續跑，LLM 會看到上一輪的 email 和 tool 結果，行為完全跑掉——
# 而且不會報錯，你只會覺得「奇怪它怎麼答非所問」。
thread_id_1 = uuid.uuid4()
thread_config_1 = {"configurable": {"thread_id": thread_id_1}}

# 這個 stream 會一路跑到第一個 interrupt() 為止，然後「正常結束」。
# ⚠️ 反直覺：graph 中斷不是拋例外，這個 for 迴圈就是安安靜靜地跑完了。
# 中斷的證據是最後一個 chunk 裡多了一個 '__interrupt__' key（LangGraph 的保留字）。
# 不去檢查它的話，你會以為 graph 已經跑完了。
print("Running the graph until the first interrupt...")
for chunk in graph.stream({"email_input": email_input_respond}, config=thread_config_1):
    # Interrupt_Object.value 就是我們在 node 裡傳給 interrupt([request]) 的那個 list，原封不動送出來。
    # 所以 .value[0] 就是那個 request dict——Agent Inbox 拿到的也正是這包東西。
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

發生了什麼事？我們觸發了 [interrupt](https://langchain-ai.github.io/langgraph/concepts/interrupts/)，它在該 tool call 處暫停了執行。你可以看到我們所中斷的 `action`（tool call 名稱）與 `args`（tool call 參數）被顯示給使用者。

那麼，我們要如何處理這個 interrupt 呢？這就是 `Command` 介面登場的地方了。[`Command` object 具備多項強大的能力](https://langchain-ai.github.io/langgraph/how-tos/command/)。我們在先前的 notebook 中曾用它來引導 graph 的流向：
- `goto`：指定接下來要路由到哪個 node
- `update`：在繼續執行前修改 state

在這裡，我們會用它從中斷的 state 恢復 graph：
- `resume`：提供要從 interrupt 呼叫回傳的值

我們可以回傳任何 graph 被設計來處理的值。在我們的情況中，graph 被設計來處理一個 dict 清單，每個 dict 只有一個 key `type`，其值可能是 `accept`、`edit`、`ignore` 或 `response`。所以我們只要把 `{"type": "accept"}` 傳給 `resume` 參數，就能告訴 graph 我們接受這個 tool call。

In [ ]:
from langgraph.types import Command

print(f"\nSimulating user accepting the {Interrupt_Object.value[0]['action_request']} tool call...")
# ⚠️ 全章最反直覺的一行：stream 的第一個參數不是「新的輸入」，而是 Command(resume=...)。
# 它的意思是「別重跑，把這個值塞回上次停住的那個 interrupt()，從那裡繼續」。
#
# 對照組：interrupt_before 那一套是用 stream(None, ...) 續跑——傳 None 代表「沒有新輸入」；
# 這裡則要傳一個 Command 物件，因為我們有東西要「餵回去」給 interrupt() 當回傳值。
#
# resume 是 list，對應 node 裡 interrupt([request]) 的那個 list（一進一出、長度與順序要對得起來）。
# {"type": "accept"} 正是使用者在 Agent Inbox 按下「同意」時，UI 幫你送出來的東西。
#
# ⚠️ 恢復之後，interrupt_handler 是「從第一行整個重跑」，不是接著跑（詳見該 node 的註解）。
for chunk in graph.stream(Command(resume=[{"type": "accept"}]), config=thread_config_1):
    # 恢復之後 graph 繼續往下跑，撞到「下一張工單」的 interrupt 又停住。
    # 所以這個迴圈長得跟上一個 cell 幾乎一樣：一次 stream = 跑到下一個暫停點為止。
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

In [ ]:
# 和上一個 cell 一字不差，只是又按了一次「同意」——這次同意的是下一張工單（write_email）。
# 一張工單一次 interrupt，所以有幾張要審，就得 resume 幾次。
# ⚠️ 這也是 HITL 的真實成本：agent 每多一個要審的動作，人就多被打斷一次。
#    hitl_tools 那份清單放得越寬，assistant 就越安全，但也越不像「自動」助理。
print(f"\nSimulating user accepting the {Interrupt_Object.value[0]['action_request']} tool call...")
for chunk in graph.stream(Command(resume=[{"type": "accept"}]), config=thread_config_1):
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

In [ ]:
# 從 checkpoint 把最終的完整對話撈出來。
# 看點：被 accept 的工單，參數和 LLM 原本開的一模一樣——人類介入沒有在歷史上留下痕跡。
# 拿這份輸出和下一個 edit 示範的輸出對照著看，最能看出「竄改歷史」到底改了什麼。
state = graph.get_state(thread_config_1)
for m in state.values['messages']:
    m.pretty_print()



### Interrupts 讓我們能編輯 Tool Calls

這個測試示範了在 HITL 流程中，人類的修改是如何運作的：
1. 我們從和之前相同的稅務規劃 email 開始
2. agent 用相同的參數提議一場會議
3. 這一次，使用者*編輯*了會議提案，做出以下更動：
   - 時長從 45 分鐘改為 30 分鐘
   - 把會議主旨改得更精簡
4. agent 在草擬 email 時會因應這些更動
5. 使用者進一步*編輯* email，使其更簡短、更不那麼正式
6. workflow 完成，兩項修改都被納入其中

這個情境展現了 HITL 最強大的面向之一：

* 使用者可以在 agent 的動作執行*之前*對其做出精準的修改，確保最終結果符合自己的偏好，而不必親自處理所有的細節。

In [ ]:
# 與先前相同的 email
email_input_respond = {
    "to": "Lance Martin <lance@company.com>",
    "author": "Project Manager <pm@client.com>",
    "subject": "Tax season let's schedule call",
    "email_thread": "Lance,\n\nIt's tax season again, and I wanted to schedule a call to discuss your tax planning strategies for this year. I have some suggestions that could potentially save you money.\n\nAre you available sometime next week? Tuesday or Thursday afternoon would work best for me, for about 45 minutes.\n\nRegards,\nProject Manager"
}

# 第二個示範：edit。同一封信、同一張 graph，只有人類的選擇不同。
# ⚠️ 注意這裡連 checkpointer 都重新 new 一個，不只是換 thread_id。
# 其實光換 thread_id 就足以隔離了（不同存檔）；重 new 是為了讓每個示範完全獨立、可以單獨重跑。
checkpointer = InMemorySaver()
graph = overall_workflow.compile(checkpointer=checkpointer)
thread_id_2 = uuid.uuid4()
thread_config_2 = {"configurable": {"thread_id": thread_id_2}}

# 執行 graph 直到第一次 interrupt - 會被分類為 "respond"，且 agent 會建立一個 write_email 的 tool call
print("Running the graph until the first interrupt...")
for chunk in graph.stream({"email_input": email_input_respond}, config=thread_config_2):
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

編輯 `schedule_meeting` 的 tool call

當 agent 提議初步的會議排程時，我們現在模擬使用者透過編輯功能做出修改。這示範了 `edit` 回應類型的運作方式：

1. 使用者收到和前一個測試相同的會議提案
2. 他們不接受，而是修改參數：
   - 把時長從 45 分鐘縮短為 30 分鐘
   - 維持同樣的日期與時間
3. `edit` 回應會包含整組修改後的參數
4. interrupt handler 會用這些編輯後的參數取代原本的 tool 參數
5. 該 tool 會以使用者的修改來執行

這顯示了編輯功能如何讓使用者精準掌控 agent 的動作，同時仍讓 agent 處理執行的細節。

In [ ]:
# 現在模擬使用者編輯 schedule_meeting 的 tool call。
#
# ⚠️ 踩雷一：edit 必須給「整組」參數，不是只給你想改的那個。
# 這裡明明只想把 45 分鐘改成 30，但 attendees / subject / preferred_day / start_time 全都得重打一次。
# 因為 interrupt_handler 是拿這包東西「整包覆蓋」原本的 args，不是 merge——
# 少給一個，那個參數就直接消失了。
print("\nSimulating user editing the schedule_meeting tool call...")
edited_schedule_args = {
    "attendees": ["pm@client.com", "lance@company.com"],
    "subject": "Tax Planning Discussion",
    "duration_minutes": 30,  # 由 45 改為 30
    "preferred_day": "2025-05-06",
    "start_time": 14 
}

# ⚠️ 踩雷二：args 包了兩層——"args": {"args": edited_schedule_args}
# 外層 args 是「Agent Inbox 回應物件」的欄位，內層 args 才是「要覆蓋掉的 tool call 參數」。
# 這就是 interrupt_handler 裡 edited_args = response["args"]["args"] 兩個 ["args"] 的由來。
# 對照組：response 類型只有一層——{"type": "response", "args": "一句話"}，args 直接就是字串。
for chunk in graph.stream(Command(resume=[{"type": "edit", "args": {"args": edited_schedule_args}}]), config=thread_config_2):
    # 檢視 response_agent 最新的一則 message
    if 'response_agent' in chunk:
        chunk['response_agent']['messages'][-1].pretty_print()
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

編輯 `write_email` 的 tool call

在接受修改後的會議排程之後，agent 草擬了一封反映 30 分鐘時長的 email。現在我們示範編輯在 email 內容上是如何運作的：

1. agent 已調整其 email 以提及較短的 30 分鐘時長
2. 我們模擬使用者想對這封 email 做出更顯著的更動：
   - 完全重寫內容，使其更簡短、更不正式
   - 更改 email 中提到的會議日期（展示使用者如何能修正 agent 的錯誤）
   - 改成請求對方確認，而非把會議陳述為已成定局
3. `edit` 回應包含整封全新的 email 內容
4. tool 的參數會以這份編輯後的內容更新
5. email 會以使用者偏好的措辭寄出

這個範例展現了 HITL 在複雜溝通任務上的威力 — agent 處理結構與初步內容，而人類則能精修語氣、風格與實質內容。

In [ ]:
# 同樣是 edit，這次改的是 email 內容，機制和上一個 cell 完全一樣。
# 唯一值得停下來講的：使用者順手改掉了 agent 寫錯的會議日期。
# 這正是 HITL 最實在的價值——錯誤是在信寄出去「之前」被攔下來的，
# 而不是事後道歉。agent 負責把 90% 的字打完，人負責攔住那 10%。
print("\nSimulating user editing the write_email tool call...")
edited_email_args = {
    "to": "pm@client.com",
    "subject": "Re: Tax season let's schedule call",
    "content": "Hello Project Manager,\n\nThank you for reaching out about tax planning. I scheduled a 30-minute call next Thursday at 3:00 PM. Would that work for you?\n\nBest regards,\nLance Martin"
}

for chunk in graph.stream(Command(resume=[{"type": "edit", "args": {"args": edited_email_args}}]), config=thread_config_2):
    # 檢視 response_agent 最新的一則 message
    if 'response_agent' in chunk:
        chunk['response_agent']['messages'][-1].pretty_print()
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

看看完整的訊息歷史，並查看 trace，以檢視編輯後的 tool calls：

https://smith.langchain.com/public/21769510-d57a-41e4-b5c7-0ddb23c237d8/r

In [ ]:
state = graph.get_state(thread_config_2)
for m in state.values['messages']:
    m.pretty_print()

### Interrupts 讓我們能對 Tool Calls 提供回饋

這組測試示範了「response」這項能力 — 提供回饋，而不是編輯或接受：

1. 首先，我們測試針對會議排程的回饋：
   - 使用者提供具體偏好（30 分鐘而非 45 分鐘，以及偏好下午的會議）
   - agent 把這份回饋納入一份修訂後的提案
   - 使用者接著接受修訂後的會議排程

2. 其次，我們測試針對 email 草擬的回饋：
   - 使用者要求一封更簡短、更不正式、並帶有特定結尾語句的 email
   - agent 依此指引完全重寫了該 email
   - 使用者接受新的草稿

3. 最後，我們測試針對問題的回饋：
   - 對於那封早午餐邀約，使用者帶著額外情境回答了問題
   - agent 運用這項資訊草擬出一份合適的 email 回應
   - workflow 帶著使用者整合進來的輸入繼續進行

「response」這項能力在接受與編輯之間搭起了橋樑 — 使用者可以引導 agent，而不必親自撰寫完整的內容。這在以下情況下特別強大：
- 調整語氣與風格
- 補上 agent 遺漏的情境
- 重新導引 agent 的做法
- 以一種能形塑後續步驟的方式回答問題

In [ ]:
# Respond - 會議邀約的 email
email_input_respond = {
    "to": "Lance Martin <lance@company.com>",
    "author": "Project Manager <pm@client.com>",
    "subject": "Tax season let's schedule call",
    "email_thread": "Lance,\n\nIt's tax season again, and I wanted to schedule a call to discuss your tax planning strategies for this year. I have some suggestions that could potentially save you money.\n\nAre you available sometime next week? Tuesday or Thursday afternoon would work best for me, for about 45 minutes.\n\nRegards,\nProject Manager"
}

# 第三個示範：response（給回饋）。一樣是全新存檔。
# （thread_id 從 2 直接跳到 5，是教材刪過 cell 留下的殘跡，數字本身沒有意義。）
checkpointer = InMemorySaver()
graph = overall_workflow.compile(checkpointer=checkpointer)
thread_id_5 = uuid.uuid4()
thread_config_5 = {"configurable": {"thread_id": thread_id_5}}

# 執行 graph 直到第一次 interrupt
# email 會被分類為 "respond"
# agent 會建立 schedule_meeting 與 write_email 的 tool call
print("Running the graph until the first interrupt...")
for chunk in graph.stream({"email_input": email_input_respond}, config=thread_config_5):
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

為 `schedule_meeting` 的 tool call 提供回饋

現在我們來探索會議排程的回饋能力：

1. agent 提議標準的 45 分鐘會議，時間在週二下午 2:00
2. 我們不接受也不編輯，而是用自然語言提供回饋
3. 我們的回饋指明兩項偏好：
   - 較短的會議（30 分鐘而非 45 分鐘）
   - 偏好下午的會議（2pm 之後）
4. agent 透過 `response` 類型收到這份回饋
5. interrupt handler 會把這份回饋作為一則 message 加入 state
6. agent 處理這份回饋，並產生一個納入這些偏好的新 tool call

與需要指明整組參數的直接編輯不同，回饋讓使用者能以對話的方式表達自己的偏好。接著 agent 必須詮釋這份回饋，並適切地加以運用以產生一份修訂後的提案。

In [ ]:
# response 的 args 是「一句自然語言」，不是參數 dict——這是它和 edit 最大的差別。
#   edit     ：你自己把參數填好，agent 照做，LLM 沒有再開口的機會。
#   response ：你只講方向，回到 llm_call，讓 LLM 自己重新開一張工單。
#
# ⚠️ 所以兩者「下一個 interrupt」的性質完全不同：
#   response 之後冒出來的，是「同一件事的修訂版」（重排過的會議，還要再審一次）。
#   edit     之後冒出來的，是「下一件事」（會議當場就排掉了，接著要審寫信）。
# 換句話說 response 比較貴：多跑一次 LLM、多打斷人一次，換來的是不必自己填參數。
print(f"\nSimulating user providing feedback for the {Interrupt_Object.value[0]['action_request']['action']} tool call...")
for chunk in graph.stream(Command(resume=[{"type": "response", "args": "Please schedule this for 30 minutes instead of 45 minutes, and I prefer afternoon meetings after 2pm."}]), config=thread_config_5):
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

在提供回饋之後，接受 `schedule_meeting` 的 tool call

In [ ]:
# LLM 讀了回饋、重開了一張工單，這次才按同意。
# 一次 response 通常要配一次 accept：講方向 → 看新版 → 放行。
print(f"\nSimulating user accepting the {Interrupt_Object.value[0]['action_request']} tool call...")
for chunk in graph.stream(Command(resume=[{"type": "accept"}]), config=thread_config_5):
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

現在為 `write_email` 的 tool call 提供回饋

在接受修訂後的會議排程之後，agent 草擬了一封 email。我們現在來測試針對 email 內容的回饋：

1. agent 的 email 相對正式且詳盡
2. 我們提供風格上的回饋，要求：
   - 一封更簡短、更精簡的 email
   - 較不正式的語氣
   - 一句關於期待這場會議的特定結尾語句
3. agent 處理這份回饋，並完全重寫了這封 email
4. 新的草稿更短、更隨性，並包含所要求的結尾

這示範了自然語言回饋在內容創作上的威力：
- 使用者不必親自重寫整封 email
- 他們可以對風格、語氣與內容提供高層次的指引
- agent 依此指引處理實際的撰寫
- 結果能更貼合使用者的偏好，同時保留必要的資訊

訊息歷史會同時顯示原始與修訂後的 email，清楚呈現回饋是如何被納入的。

In [ ]:
# 同樣的 response 手法，這次用在 email 內容上：不自己動筆，只講風格要求，讓 LLM 重寫。
# 這是 response 相對 edit 最划算的場景——要改的是語氣和長度，用講的比自己重打整封信快。
print(f"\nSimulating user providing feedback for the {Interrupt_Object.value[0]['action_request']['action']} tool call...")
for chunk in graph.stream(Command(resume=[{"type": "response", "args": "Shorter and less formal. Include a closing statement about looking forward to the meeting!"}]), config=thread_config_5):
    # 檢視 response_agent 最新的一則 message
    if 'response_agent' in chunk:
        chunk['response_agent']['messages'][-1].pretty_print()
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

在提供回饋之後，接受 `write_email` 的 tool call

In [ ]:
print(f"\nSimulating user accepting the {Interrupt_Object.value[0]['action_request']} tool call...")
for chunk in graph.stream(Command(resume=[{"type": "accept"}]), config=thread_config_5):
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

看看完整的訊息歷史，並查看 trace：

https://smith.langchain.com/public/57006770-6bb3-4e40-b990-143c373ebe60/r

我們可以看到使用者的回饋被納入了 tool calls 中。

In [ ]:
state = graph.get_state(thread_config_5)
for m in state.values['messages']:
    m.pretty_print()

### Interrupts 讓新的 Tools 成為可能

現在我們來試試一封會呼叫 `Question` tool 來提供回饋的 email

最後，我們測試回饋在 `Question` tool 上是如何運作的：

1. 對於那封早午餐邀約 email，agent 詢問偏好的日期與時間
2. 我們不忽略，而是提供一則帶有額外情境的實質回應：
   - 確認我們想邀請信中提到的那些人
   - 註明我們需要確認哪個週末最合適
   - 補充需要訂位的相關資訊
3. agent 運用這項資訊來：
   - 草擬一份納入我們所有回饋的完整 email 回應
   - 注意到我們並未提供具體的日期／時間，因此建議查看行事曆
   - 把訂位的細節納入其中
4. 這封完整的 email 同時反映了原始請求與我們額外的指引

這示範了問題的回應如何能形塑整個 workflow：
- 問題讓 agent 能蒐集缺少的資訊
- 使用者的回應可以同時包含直接的答案與額外的情境
- agent 把所有這些資訊整合進它接下來的動作
- 最終結果反映了人類與 AI 兩者協作的智慧

In [ ]:
# Respond
email_input_respond = {
    "to": "Lance Martin <lance@company.com>",
    "author": "Partner <partner@home.com>",
    "subject": "Dinner?",
    "email_thread": "Hey, do you want italian or indian tonight?"}

# 第四個示範：Question tool。
# 這封信 agent 根本沒有足夠資訊可以回（要義大利菜還是印度菜？只有人類知道），
# 所以它會開一張 Question 工單。
# 這是 interrupt 帶來的新能力：agent 卡住時可以「反問」，而不是硬掰一個答案。
# ⚠️ 前提揭露：沒有 HITL 的 agent 之所以會胡說八道，往往不是因為它笨，
#    而是因為流程根本沒給它「我不知道」這個選項——它只能猜。
checkpointer = InMemorySaver()
graph = overall_workflow.compile(checkpointer=checkpointer)
thread_id_6 = uuid.uuid4()
thread_config_6 = {"configurable": {"thread_id": thread_id_6}}

# 執行 graph 直到第一次 interrupt
print("Running the graph until the first interrupt...")
for chunk in graph.stream({"email_input": email_input_respond}, config=thread_config_6):
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

為 `Question` 的 tool call 提供回饋

In [ ]:
# 回答 Question 用的是 response 類型——因為 Question 的 config 只開了 ignore／respond 兩顆按鈕。
# 使用者的答案會變成一則 tool message 回到 llm_call，LLM 拿去寫信。
# 注意 Question 這個 tool 從頭到尾沒有被 invoke 過一次：它的「執行」就是問到答案這件事本身。
print(f"\nSimulating user providing feedback for the {Interrupt_Object.value[0]['action_request']['action']} tool call...")
for chunk in graph.stream(Command(resume=[{"type": "response", "args": "Let's do indian."}]), config=thread_config_6):
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

接受 `write_email` 的 tool call

In [ ]:
# LLM 拿到答案後開了 write_email 工單，這裡放行。
# 這一輪走完，四種 response type（accept／edit／ignore／response）就全部示範過了。
print(f"\nSimulating user accepting the {Interrupt_Object.value[0]['action_request']['action']} tool call...")
for chunk in graph.stream(Command(resume=[{"type": "accept"}]), config=thread_config_6):
    # 檢視 response_agent 最新的一則 message
    if 'response_agent' in chunk:
        chunk['response_agent']['messages'][-1].pretty_print()
    # 若有 interrupt 物件則加以檢視
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

看看完整的訊息歷史，並查看 trace：

https://smith.langchain.com/public/f4c727c3-b1d9-47a5-b3d0-3451619db8a2/r

我們可以看到使用者的回饋被納入了 email 回應中。

In [ ]:
state = graph.get_state(thread_config_6)
for m in state.values['messages']:
    m.pretty_print()

### Deployment

我們來從 `src/email_assistant/email_assistant_hitl.py` 建立一個具備 HITL 的 email assistant 本機部署。
 
和之前一樣，執行 `langgraph dev`，在 Studio 中選擇 `email_assistant_hitl`，然後送出這封 email：

In [ ]:
{
  "author": "Alice Smith <alice.smith@company.com>",
  "to": "John Doe <john.doe@company.com>",
  "subject": "Quick question about API documentation",
  "email_thread": "Hi John,\nI was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?\nSpecifically, I'm looking at:\n- /auth/refresh\n- /auth/validate\nThanks!\nAlice"
}

我們的伺服器是無狀態（stateless）的。在本機部署時，threads 只是被保存到本機檔案系統（專案資料夾中的 `.langgraph_api`）。

若採用[託管式（hosted）](https://langchain-ai.github.io/langgraph/tutorials/deployment/#other-deployment-options)部署，threads 則會儲存在 Postgres 中。

被中斷的 threads 是狀態為 'interrupted' 的 threads，我們可以在 Studio 中看到該次 interrupt：

![studio-img](img/studio-interrupt.png)

我們會使用一個自訂介面來檢視這些被中斷的 threads，也就是 [Agent Inbox](https://dev.agentinbox.ai/)。

這個介面是個很好用的方式，可以對 LangGraph agent 所採取的特定動作進行編輯、核准、忽略或提供回饋。

如果你前往 [dev.agentinbox.ai](https://dev.agentinbox.ai/)，就能輕鬆連接到這張 graph：
   * Graph name：來自 `langgraph.json` 檔案的名稱（`email_assistant_hitl`）
   * Graph URL：`http://127.0.0.1:2024/`

屆時所有被中斷的 threads 執行都會顯示出來：

![agent-inbox-img](img/agent-inbox.png)

Agent Inbox 其實就是使用一個帶有 `resume` 的 `Command` 來恢復 graph，正如上面[在 SDK 中所示](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/wait-user-input/#interacting-with-the-agent)。